# Rotation validation: Vo centre finding

Validate six original Vo-method checks across I12 datasets 1 and 2, I13 dataset 2,
GEANT4 dataset 1 and K11 dataset 1. The I12 dataset 2 central-row and row-averaging
cases share preprocessing. Inputs are corrected transmission (no negative logarithm).
Centre coordinates are in pixels of the original detector width.

Transferred from `zenodo-tests/test_recon/test_rotation.py` (source SHA-256:
`a6e5bfd58d3962fff42f2c044426e2a2f8be7731e6c6867b779ed037572049a4`). The numerical reference values and dtype checks are retained;
plots support interpretation and do not replace regression assertions.

## Data and execution

Use the named archives from [Zenodo record 17252190](https://zenodo.org/records/17252190).
Set `HTTOMOLIBGPU_DATA_DIR` for an archive directory other than the checkout's `Zenodo` folder.
Run from a compatible HTTomolibGPU source checkout with CuPy and a CUDA GPU:

```bash
python -m pip install -e '.[validation]'
python .scripts/download_zenodo.py Zenodo
python -m pytest --nbmake --nbmake-timeout=1800 --overwrite docs/source/validation/rotation/center_vo.ipynb
```

CUDA toolkit headers must be discoverable. In a Linux Conda environment, if
`$CONDA_PREFIX/targets/x86_64-linux/include/cuda_runtime.h` exists, set
`CUDA_PATH=$CONDA_PREFIX/targets/x86_64-linux` before running the notebook.

Run GPU notebooks serially. Documentation renders saved outputs without GPU execution.
This notebook reports all numerical checks together after displaying each case;
a regression is still a failing assertion, with its actual result retained in the outputs.


In [ ]:
import gc
import hashlib
import json
import os
import platform
from importlib.metadata import version
from pathlib import Path

import cupy as cp
import matplotlib.pyplot as plt
import numpy as np

from httomolibgpu.prep.normalize import dark_flat_field_correction, minus_log
from httomolibgpu.recon.rotation import find_center_vo, find_center_pc, find_center_360

repo_root = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "httomolibgpu").is_dir()
    ),
    None,
)
configured_dir = os.environ.get("HTTOMOLIBGPU_DATA_DIR")
if configured_dir:
    data_dir = Path(configured_dir).expanduser()
elif repo_root is not None:
    data_dir = repo_root / "Zenodo"
else:
    raise FileNotFoundError("Set HTTOMOLIBGPU_DATA_DIR to the Zenodo archive directory")

props = cp.cuda.runtime.getDeviceProperties(cp.cuda.Device().id)
environment = {
    "python": platform.python_version(),
    "packages": {
        name: version(name) for name in ("httomolibgpu", "numpy", "matplotlib")
    },
    "cupy": cp.__version__,
    "cuda_runtime": cp.cuda.runtime.runtimeGetVersion(),
    "cuda_driver": cp.cuda.runtime.driverGetVersion(),
    "gpu": (
        props["name"].decode() if isinstance(props["name"], bytes) else props["name"]
    ),
}
print(json.dumps(environment, indent=2))
results = []
dataset_provenance = {}


def clean_memory():
    gc.collect()
    cp.get_default_memory_pool().free_all_blocks()
    cp.get_default_pinned_memory_pool().free_all_blocks()


def archive_path(name):
    path = data_dir / (name + ".npz")
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing {path}; run .scripts/download_zenodo.py first"
        )
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    dataset_provenance[name] = {
        "archive": path.name,
        "sha256": digest.hexdigest(),
        "zenodo_record": "17252190",
    }
    print(json.dumps(dataset_provenance[name], indent=2))
    return path


def load_transmission(name):
    with np.load(archive_path(name)) as archive:
        raw = cp.asarray(archive["projdata"])
        flats = cp.asarray(archive["flats"])
        darks = cp.asarray(archive["darks"])
        print("Raw, flats, darks:", raw.shape, flats.shape, darks.shape)
    transmission = dark_flat_field_correction(raw, flats, darks)
    del raw, flats, darks
    clean_memory()
    return transmission


def display_sinogram(sinogram, observed, expected, title):
    image = cp.asnumpy(sinogram)
    lo, hi = np.percentile(image[np.isfinite(image)], [1, 99])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    im = axes[0].imshow(image, cmap="gray", aspect="auto", vmin=lo, vmax=hi)
    axes[0].axvline(float(observed), color="tab:orange", label="Observed centre")
    axes[0].axvline(
        float(expected), color="tab:cyan", linestyle="--", label="Reference centre"
    )
    axes[0].set(
        title=title, xlabel="Detector column (pixel)", ylabel="Projection index"
    )
    axes[0].legend(fontsize=8)
    fig.colorbar(im, ax=axes[0], label="Corrected transmission")
    axes[1].plot(np.mean(image, axis=0))
    axes[1].axvline(float(observed), color="tab:orange")
    axes[1].axvline(float(expected), color="tab:cyan", linestyle="--")
    axes[1].set(
        title="Angular mean profile",
        xlabel="Detector column (pixel)",
        ylabel="Mean transmission",
    )
    plt.show()
    plt.close(fig)


def check_results():
    print(json.dumps(results, indent=2))
    failed = [row for row in results if not row["passed"]]
    assert not failed, "Original rotation regressions failed: " + json.dumps(
        failed, indent=2
    )
    print(f"All {len(results)} original rotation checks passed.")

## I12 DATASET1

Original case: `test_center_vo_i12_dataset1`. Normalise the complete archive using default dark/flat
correction, then pass its middle detector row as a two-dimensional sinogram.
Expected centre: **1253.75 pixels**, with dtype `numpy.float32`.
Parameters: `{'smin': -100, 'smax': 100, 'step': 0.25}` (an empty mapping retains the original defaults).


In [ ]:
data = load_transmission("i12_dataset1")
mid_row = data.shape[1] // 2
parameters = {"smin": -100, "smax": 100, "step": 0.25}
cor = find_center_vo(data[:, mid_row, :], **parameters)
passed = bool(cor == 1253.75 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_vo_i12_dataset1",
        "dataset": "i12_dataset1",
        "row": mid_row,
        "parameters": parameters,
        "observed": float(cor),
        "expected": 1253.75,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
display_sinogram(data[:, mid_row, :], cor, 1253.75, "i12_dataset1: central row")

In [ ]:
del data
clean_memory()

## I12 DATASET2

Original case: `test_center_vo_i12_dataset2`. Normalise the complete archive using default dark/flat
correction, then pass its middle detector row as a two-dimensional sinogram.
Expected centre: **1197.75 pixels**, with dtype `numpy.float32`.
Parameters: `{'smin': -100, 'smax': 100, 'step': 0.25}` (an empty mapping retains the original defaults).


In [ ]:
data = load_transmission("i12_dataset2")
mid_row = data.shape[1] // 2
parameters = {"smin": -100, "smax": 100, "step": 0.25}
cor = find_center_vo(data[:, mid_row, :], **parameters)
passed = bool(cor == 1197.75 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_vo_i12_dataset2",
        "dataset": "i12_dataset2",
        "row": mid_row,
        "parameters": parameters,
        "observed": float(cor),
        "expected": 1197.75,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
display_sinogram(data[:, mid_row, :], cor, 1197.75, "i12_dataset2: central row")

### I12 dataset 2: averaging neighbouring rows

Original case: `test_center_vo_average_i12_dataset2`. Pass the exact `[:, 10:25, :]`
selection to the method with `average_radius=5`, `smin=-100`, `smax=100` and
`step=0.25`. The selection has 15 detector rows; its middle row is index 7,
so the averaging window covers original detector rows 12 through 22 inclusive.
This is a different slice selection from the central-row case, not solely a
change in averaging strength. Expected centre: **1199.25 pixels**, dtype `float32`.


In [ ]:
selection = data[:, 10:25, :]
parameters = {"average_radius": 5, "smin": -100, "smax": 100, "step": 0.25}
cor = find_center_vo(selection, **parameters)
passed = bool(cor == 1199.25 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_vo_average_i12_dataset2",
        "dataset": "i12_dataset2",
        "selection": "[:, 10:25, :]",
        "parameters": parameters,
        "observed": float(cor),
        "expected": 1199.25,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
display_sinogram(
    cp.mean(selection[:, 2:13, :], axis=1),
    cor,
    1199.25,
    "I12 dataset 2: mean of detector rows 12:23",
)
del selection

In [ ]:
del data
clean_memory()

## I13 DATASET2

Original case: `test_center_vo_i13_dataset2`. Normalise the complete archive using default dark/flat
correction, then pass its middle detector row as a two-dimensional sinogram.
Expected centre: **1286.25 pixels**, with dtype `numpy.float32`.
Parameters: `{'smin': -100, 'smax': 100, 'step': 0.25}` (an empty mapping retains the original defaults).


In [ ]:
data = load_transmission("i13_dataset2")
mid_row = data.shape[1] // 2
parameters = {"smin": -100, "smax": 100, "step": 0.25}
cor = find_center_vo(data[:, mid_row, :], **parameters)
passed = bool(cor == 1286.25 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_vo_i13_dataset2",
        "dataset": "i13_dataset2",
        "row": mid_row,
        "parameters": parameters,
        "observed": float(cor),
        "expected": 1286.25,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
display_sinogram(data[:, mid_row, :], cor, 1286.25, "i13_dataset2: central row")

In [ ]:
del data
clean_memory()

## GEANT4 DATASET1

Original case: `test_center_vo_geant4_dataset1`. Normalise the complete archive using default dark/flat
correction, then pass its middle detector row as a two-dimensional sinogram.
Expected centre: **319.5 pixels**, with dtype `numpy.float32`.
Parameters: `{}` (an empty mapping retains the original defaults).


In [ ]:
data = load_transmission("geant4_dataset1")
mid_row = data.shape[1] // 2
parameters = {}
cor = find_center_vo(data[:, mid_row, :], **parameters)
passed = bool(cor == 319.5 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_vo_geant4_dataset1",
        "dataset": "geant4_dataset1",
        "row": mid_row,
        "parameters": parameters,
        "observed": float(cor),
        "expected": 319.5,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
display_sinogram(data[:, mid_row, :], cor, 319.5, "geant4_dataset1: central row")

In [ ]:
del data
clean_memory()

## K11 DATASET1

Original case: `test_center_vo_k11_dataset1`. Normalise the complete archive using default dark/flat
correction, then pass its middle detector row as a two-dimensional sinogram.
Expected centre: **1269.25 pixels**, with dtype `numpy.float32`.
Parameters: `{'smin': -100, 'smax': 100, 'step': 0.25}` (an empty mapping retains the original defaults).


In [ ]:
data = load_transmission("k11_dataset1")
mid_row = data.shape[1] // 2
parameters = {"smin": -100, "smax": 100, "step": 0.25}
cor = find_center_vo(data[:, mid_row, :], **parameters)
passed = bool(cor == 1269.25 and cor.dtype == np.float32)
results.append(
    {
        "test": "test_center_vo_k11_dataset1",
        "dataset": "k11_dataset1",
        "row": mid_row,
        "parameters": parameters,
        "observed": float(cor),
        "expected": 1269.25,
        "dtype": str(cor.dtype),
        "expected_dtype": "float32",
        "passed": passed,
    }
)
print(json.dumps(results[-1], indent=2))
display_sinogram(data[:, mid_row, :], cor, 1269.25, "k11_dataset1: central row")

In [ ]:
del data
clean_memory()

## Original regression checks

Exact centre values and `float32` dtype requirements match the original tests.
All cases have been evaluated before this final assertion so a failure does
not hide the other datasets' observations. Differences require investigation;
the plots are not a reason to relax or replace these references.


In [ ]:
check_results()